# 📊 Base de Dados Dinâmica — inform_26 (SAL_DAT026)

Este notebook contém toda a lógica para:
1. **Criar tabelas** para cada ano automaticamente (2024, 2025, 2026)
2. **Inserir dados** sem duplicatas
3. **Inspecionar a BD** com ferramentas interativas

## ✨ Especificações
- Fonte: ficheiros `.xlsx` formato SAL_DAT026 (header na linha 2)
- Campo de data: `FCARGA`
- Chave de deduplicação: `CODEUT`
- Tabelas criadas: `inform_26_2024`, `inform_26_2025`, `inform_26_2026`

## 🚀 Modo de Usar
Executa as células **por ordem**, uma a uma (Cell → Run All também funciona)

## PASSO 1 — Importações e Configuração

In [1]:
# ==============================================================================
# IMPORTAÇÕES
# ==============================================================================
import platform
import sqlite3
import os
import glob
import pandas as pd
from datetime import datetime
from collections import defaultdict

print("✓ Bibliotecas importadas com sucesso")

✓ Bibliotecas importadas com sucesso


In [2]:
# ==============================================================================
# CONFIGURAÇÃO DE CAMINHOS
# ==============================================================================

if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
    PASTA_FICHEIROS = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_26"
elif platform.system() == 'Darwin':
    DB_PATH = "/Volumes/RR/DB/inform_27.db"
    PASTA_FICHEIROS = "/Volumes/RR/DB/inform_26"
else:
    DB_PATH = "inform_27.db"
    PASTA_FICHEIROS = "inform_26"

print(f"DB_PATH: {DB_PATH}")
print(f"PASTA_FICHEIROS: {PASTA_FICHEIROS}")
print()

# Verificar/Criar pasta se não existir
pasta = os.path.dirname(DB_PATH)
if pasta and not os.path.exists(pasta):
    os.makedirs(pasta, exist_ok=True)
    print(f"Pasta criada: {pasta}")

# Criar BD
con = sqlite3.connect(DB_PATH)
con.close()
print(f"✓ Base de dados criada/aberta em: {DB_PATH}")

DB_PATH: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db
PASTA_FICHEIROS: C:\Users\LISARR\Documents\python\01.Financeiro\inform_26

✓ Base de dados criada/aberta em: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db


In [3]:
# ==============================================================================
# DEFINIÇÕES GLOBAIS
# ==============================================================================

COLUNAS_FICHEIRO = [
    "PROPIETARIO", "TRAYECTO", "TRANSPORTISTA", "INGRESOUT", "COSTEUT",
    "MARGENUT", "PALÉSEQUI", "OCUPALETS", "PALÉSREAL", "PESO_BRUTO",
    "OCUKILOS", "CODEUT", "FCARGA", "GESTION", "DEPART",
    "PALETS TEORIC", "ESTADO UT", "TRACTORA", "REMOLQUE", "ORIGEN",
    "LOCORIGEN", "PROVORIGEN", "PAISORIGEN", "DESTINO", "LOCDESTINO",
    "PROVDESTINO", "PAISDESTINO", "SEMANA CARGA", "DIA SEMANA CARGA", "USCODE",
    "USUARIO", "TIPOFLUJO", "CAMION_TIPO", "CAMION_CAPACIDAD", "FECHA_LLEGADA",
    "FAX", "CARGAS", "DESCARGAS", "N_PARADAS", "KM",
    "€/KM", "TEMPERATURA", "NOTAS DE TMS", "CRONOGRAMA", "HUECOS_CRONO",
    "FECHA_PREFACTURA", "Nº PREFACTURA", "HORA_CARGA", "GOON",
]

# Nomes seguros para SQLite (sem espaços/caracteres especiais)
COLUNAS_SQL = [
    "PROPIETARIO", "TRAYECTO", "TRANSPORTISTA", "INGRESOUT", "COSTEUT",
    "MARGENUT", "PALESEQUI", "OCUPALETS", "PALESREAL", "PESO_BRUTO",
    "OCUKILOS", "CODEUT", "FCARGA", "GESTION", "DEPART",
    "PALETS_TEORIC", "ESTADO_UT", "TRACTORA", "REMOLQUE", "ORIGEN",
    "LOCORIGEN", "PROVORIGEN", "PAISORIGEN", "DESTINO", "LOCDESTINO",
    "PROVDESTINO", "PAISDESTINO", "SEMANA_CARGA", "DIA_SEMANA_CARGA", "USCODE",
    "USUARIO", "TIPOFLUJO", "CAMION_TIPO", "CAMION_CAPACIDAD", "FECHA_LLEGADA",
    "FAX", "CARGAS", "DESCARGAS", "N_PARADAS", "KM",
    "EUR_KM", "TEMPERATURA", "NOTAS_TMS", "CRONOGRAMA", "HUECOS_CRONO",
    "FECHA_PREFACTURA", "NUM_PREFACTURA", "HORA_CARGA", "GOON",
]

COLUNA_ORIGEM = "ficheiro_origem"
CAMPO_DATA    = "FCARGA"        # coluna no ficheiro
CAMPO_DATA_SQL = "FCARGA"       # nome na BD
CHAVE_DEDUP   = "CODEUT"        # coluna para deduplicação

# Mapeamento coluna_ficheiro → coluna_sql
MAP_COLUNAS = dict(zip(COLUNAS_FICHEIRO, COLUNAS_SQL))

print(f"✓ {len(COLUNAS_FICHEIRO)} colunas definidas")
print(f"✓ Campo de data: {CAMPO_DATA}")
print(f"✓ Chave de deduplicação: {CHAVE_DEDUP}")
print(f"✓ Coluna de origem: {COLUNA_ORIGEM}")

✓ 49 colunas definidas
✓ Campo de data: FCARGA
✓ Chave de deduplicação: CODEUT
✓ Coluna de origem: ficheiro_origem


## PASSO 2 — Funções Auxiliares

In [4]:
# ==============================================================================
# FUNÇÃO: Ler ficheiros Excel .xlsx (SAL_DAT026 — header na linha 2)
# ==============================================================================

def ler_xlsx(caminho_ficheiro):
    """
    Lê um ficheiro .xlsx no formato SAL_DAT026.
    Header está na linha 2 (índice 1 no pandas).
    Devolve (cabecalho_sql, lista_de_listas_de_valores).
    """
    try:
        df = pd.read_excel(caminho_ficheiro, header=1, dtype=str)
        df = df.where(pd.notnull(df), None)

        # Filtrar apenas as colunas que existem no ficheiro e estão mapeadas
        colunas_presentes = [c for c in COLUNAS_FICHEIRO if c in df.columns]
        colunas_sql_presentes = [MAP_COLUNAS[c] for c in colunas_presentes]

        linhas = []
        for _, row in df[colunas_presentes].iterrows():
            linhas.append([row[c] for c in colunas_presentes])

        return colunas_sql_presentes, linhas

    except Exception as e:
        print(f"    [ERRO ao ler] {e}")
        return None, None

print("✓ Função ler_xlsx() pronta")

✓ Função ler_xlsx() pronta


In [5]:
# ==============================================================================
# FUNÇÃO: Listar ficheiros Excel
# ==============================================================================

def listar_ficheiros_excel(pasta):
    """Procura todos os ficheiros .xlsx na pasta (incluindo subpastas)."""
    encontrados = glob.glob(os.path.join(pasta, "**", "*.xlsx"), recursive=True)
    vistos = set()
    ficheiros = []
    for caminho in encontrados:
        chave = os.path.normcase(os.path.abspath(caminho))
        if chave not in vistos:
            vistos.add(chave)
            ficheiros.append(caminho)
    return sorted(ficheiros)

print("✓ Função listar_ficheiros_excel() pronta")

✓ Função listar_ficheiros_excel() pronta


In [6]:
# ==============================================================================
# FUNÇÃO: Detectar anos nos ficheiros
# ==============================================================================

def detectar_anos_nos_ficheiros(pasta):
    """
    Procura todos os ficheiros .xlsx na pasta e detecta quais são os anos
    presentes nos dados (através do campo FCARGA).
    Devolve lista ordenada de anos (strings).
    """
    ficheiros = listar_ficheiros_excel(pasta)
    anos = set()

    idx_data = COLUNAS_FICHEIRO.index(CAMPO_DATA)

    for caminho in ficheiros:
        colunas_sql, linhas = ler_xlsx(caminho)
        if colunas_sql is None:
            continue
        for linha in linhas:
            if linha and idx_data < len(linha):
                valor_data = linha[idx_data]
                if valor_data and str(valor_data).strip()[:4].isdigit():
                    ano = str(valor_data).strip()[:4]
                    anos.add(ano)

    return sorted(anos)

print("✓ Função detectar_anos_nos_ficheiros() pronta")

✓ Função detectar_anos_nos_ficheiros() pronta


In [7]:
# ==============================================================================
# FUNÇÃO: Detectar anos na BD
# ==============================================================================

def detectar_anos_na_bd(db_path):
    """
    Detecta quais são as tabelas de anos presentes na BD.
    Devolve lista ordenada de anos (strings).
    """
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tabelas = [nome for (nome,) in cur.fetchall()]
    con.close()

    anos = []
    for tabela in tabelas:
        if tabela.startswith("inform_26_"):
            ano = tabela.replace("inform_26_", "")
            if ano.isdigit() and len(ano) == 4:
                anos.append(ano)

    return sorted(anos)

print("✓ Função detectar_anos_na_bd() pronta")

✓ Função detectar_anos_na_bd() pronta


## PASSO 3 — Criar Tabelas Dinamicamente

In [8]:
# ==============================================================================
# DETECTAR ANOS NOS FICHEIROS
# ==============================================================================

print("A detectar anos nos ficheiros...")
anos_detectados = detectar_anos_nos_ficheiros(PASTA_FICHEIROS)

print()
if anos_detectados:
    print(f"✓ Anos detectados: {', '.join(anos_detectados)}")
else:
    print("[AVISO] Nenhum ano detectado. Verifica se:")
    print("  - Os ficheiros .xlsx existem em:", PASTA_FICHEIROS)
    print("  - Têm a coluna FCARGA")
    print("  - FCARGA tem valores no formato AAAAMMDD")

A detectar anos nos ficheiros...


C:\Users\LISARR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\LISARR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\LISARR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\LISARR\AppData\Local\Packages\P


✓ Anos detectados: 2024, 2025, 2026


C:\Users\LISARR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [9]:
# ==============================================================================
# CRIAR TABELAS PARA CADA ANO DETECTADO
# ==============================================================================

if not anos_detectados:
    print("[ERRO] Sem anos para processar. Verifica a pasta de ficheiros.")
else:
    colunas_sql_def = ",\n    ".join(f'"{c}" TEXT' for c in COLUNAS_SQL)

    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    print(f"A criar {len(anos_detectados)} tabela(s)...")
    print()

    for ano in anos_detectados:
        tabela_nome = f"inform_26_{ano}"
        cur.execute(f'''
            CREATE TABLE IF NOT EXISTS "{tabela_nome}" (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                {colunas_sql_def},
                "{COLUNA_ORIGEM}" TEXT
            )
        ''')
        # Índice único por CODEUT para deduplicação
        cur.execute(f'''
            CREATE UNIQUE INDEX IF NOT EXISTS "idx_inform_26_{ano}_codeut"
            ON "{tabela_nome}" ("{CHAVE_DEDUP}")
        ''')
        print(f"  ✓ Tabela {tabela_nome} criada (ou já existia).")

    con.commit()
    con.close()

    print()
    print("✓ Todas as tabelas prontas!")

A criar 3 tabela(s)...

  ✓ Tabela inform_26_2024 criada (ou já existia).
  ✓ Tabela inform_26_2025 criada (ou já existia).
  ✓ Tabela inform_26_2026 criada (ou já existia).

✓ Todas as tabelas prontas!


In [10]:
# ==============================================================================
# CONFIRMAR: Listar tabelas criadas
# ==============================================================================

con = sqlite3.connect(DB_PATH)
cur = con.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tabelas_existentes = [r[0] for r in cur.fetchall()]
con.close()

print(f"Tabelas na BD ({len(tabelas_existentes)}):")
for t in tabelas_existentes:
    print(f"  • {t}")

Tabelas na BD (18):
  • cargas_2023
  • cargas_2024
  • cargas_2025
  • cargas_2026
  • entregas_2025
  • entregas_2026
  • inform_26_2024
  • inform_26_2025
  • inform_26_2026
  • inform_27_2023
  • inform_27_2024
  • inform_27_2025
  • inform_27_2026
  • km_diario_2023
  • km_diario_2024
  • km_diario_2025
  • km_diario_2026
  • sqlite_sequence


## PASSO 4 — Inserir Dados

In [11]:
# ==============================================================================
# INSERIR DADOS DE TODOS OS FICHEIROS
# ==============================================================================

if not anos_detectados:
    print("[ERRO] Corre o Passo 3 primeiro.")
else:
    ficheiros = listar_ficheiros_excel(PASTA_FICHEIROS)

    if not ficheiros:
        print(f"[AVISO] Nenhum ficheiro .xlsx encontrado em: {PASTA_FICHEIROS}")
    else:
        print(f"Ficheiros a processar: {len(ficheiros)}")
        print()

        # Preparar SQLs de inserção por ano
        sql_por_ano = {}
        for ano in anos_detectados:
            tabela = f"inform_26_{ano}"
            placeholders = ", ".join(["?"] * (len(COLUNAS_SQL) + 1))
            cols = ", ".join(f'"{c}"' for c in COLUNAS_SQL) + f', "{COLUNA_ORIGEM}"'
            sql_por_ano[ano] = (
                f'INSERT OR IGNORE INTO "{tabela}" ({cols}) VALUES ({placeholders})'
            )

        # Índice da coluna de data na lista COLUNAS_SQL
        idx_data_sql = COLUNAS_SQL.index(CAMPO_DATA_SQL)

        totais_inseridos = defaultdict(int)
        totais_duplicados = 0
        totais_sem_data = 0

        con = sqlite3.connect(DB_PATH)
        cur = con.cursor()

        for caminho in ficheiros:
            nome_ficheiro = os.path.basename(caminho)
            print(f"→ {nome_ficheiro}")

            colunas_sql_lidas, linhas = ler_xlsx(caminho)
            if colunas_sql_lidas is None or not linhas:
                print("    [AVISO] Sem dados ou erro de leitura.")
                print()
                continue

            # Organizar linhas por ano
            batches_por_ano = {ano: [] for ano in anos_detectados}
            sem_data = 0

            for linha in linhas:
                # Ignorar linhas completamente vazias
                if all(v is None or str(v).strip() == "" for v in linha):
                    continue

                # Remapear para a ordem fixa de COLUNAS_SQL
                mapa = dict(zip(colunas_sql_lidas, linha))
                valores_ordenados = [mapa.get(c) for c in COLUNAS_SQL]

                # Extrair e validar data
                valor_data = valores_ordenados[idx_data_sql]
                if not valor_data or not str(valor_data).strip()[:4].isdigit():
                    sem_data += 1
                    continue

                ano = str(valor_data).strip()[:4]

                if ano in batches_por_ano:
                    batches_por_ano[ano].append(valores_ordenados + [nome_ficheiro])
                else:
                    sem_data += 1

            # Inserir batches por ano
            for ano in anos_detectados:
                batch = batches_por_ano[ano]
                if not batch:
                    continue
                antes = con.total_changes
                cur.executemany(sql_por_ano[ano], batch)
                con.commit()
                inseridos = con.total_changes - antes
                totais_inseridos[ano] += inseridos
                totais_duplicados += len(batch) - inseridos

            totais_sem_data += sem_data

            resumo = ", ".join(
                f"{totais_inseridos[a]} novo(s) em {a}" for a in sorted(anos_detectados)
            )
            print(f"  -> {resumo}")
            print(f"     {totais_duplicados} duplicata(s), {totais_sem_data} sem data")
            print()

        con.close()

        print("=" * 70)
        print("RESUMO FINAL")
        print("=" * 70)
        print()
        print(f"Ficheiros processados: {len(ficheiros)}")
        print()
        for ano in sorted(anos_detectados):
            print(f"  Linhas novas inseridas em {ano}: {totais_inseridos[ano]:,}")
        print()
        print(f"Linhas já existentes (duplicadas por CODEUT): {totais_duplicados:,}")
        print(f"Linhas sem FCARGA válido: {totais_sem_data:,}")
        print("=" * 70)
        print()
        print("✓ Inserção de dados concluída!")

C:\Users\LISARR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Ficheiros a processar: 8

→ SAL_DAT026 (2).xlsx
  -> 0 novo(s) em 2024, 0 novo(s) em 2025, 5448 novo(s) em 2026
     0 duplicata(s), 0 sem data

→ SAL_DAT026 (3).xlsx
  -> 0 novo(s) em 2024, 30 novo(s) em 2025, 5448 novo(s) em 2026
     0 duplicata(s), 0 sem data

→ SAL_DAT026 (4).xlsx


C:\Users\LISARR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\LISARR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  -> 0 novo(s) em 2024, 14239 novo(s) em 2025, 5448 novo(s) em 2026
     0 duplicata(s), 0 sem data

→ SAL_DAT026 (5).xlsx


C:\Users\LISARR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


...

C:\Users\LISARR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\LISARR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  -> 16983 novo(s) em 2024, 23929 novo(s) em 2025, 5448 novo(s) em 2026
     0 duplicata(s), 0 sem data

→ SAL_DAT026 (8).xlsx


C:\Users\LISARR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  -> 27388 novo(s) em 2024, 23929 novo(s) em 2025, 5448 novo(s) em 2026
     0 duplicata(s), 0 sem data

→ SAL_DAT026.xlsx
    [AVISO] Sem dados ou erro de leitura.

RESUMO FINAL

Ficheiros processados: 8

  Linhas novas inseridas em 2024: 27,388
  Linhas novas inseridas em 2025: 23,929
  Linhas novas inseridas em 2026: 5,448

Linhas já existentes (duplicadas por CODEUT): 0
Linhas sem FCARGA válido: 0

✓ Inserção de dados concluída!


C:\Users\LISARR\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


## PASSO 5 — Inspecionar a Base de Dados

In [12]:
# ==============================================================================
# ESTATÍSTICAS POR TABELA/ANO
# ==============================================================================

print("=" * 70)
print("INSPEÇÃO DA BASE DE DADOS")
print("=" * 70)
print()

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

anos_na_bd = detectar_anos_na_bd(DB_PATH)
print(f"Anos com tabelas na BD: {', '.join(anos_na_bd)}")
print()

total_geral = 0
for ano in anos_na_bd:
    tabela = f"inform_26_{ano}"

    cur.execute(f'SELECT COUNT(*) FROM "{tabela}"')
    count = cur.fetchone()[0]
    total_geral += count

    cur.execute(f'SELECT COUNT(DISTINCT ficheiro_origem) FROM "{tabela}"')
    num_ficheiros = cur.fetchone()[0]

    cur.execute(f'SELECT MIN(FCARGA), MAX(FCARGA) FROM "{tabela}"')
    row = cur.fetchone()
    data_min = row[0] if row[0] else "N/A"
    data_max = row[1] if row[1] else "N/A"

    cur.execute(f'SELECT COUNT(DISTINCT CODEUT) FROM "{tabela}"')
    codeut_unicos = cur.fetchone()[0]

    print(f"📊 inform_26_{ano}:")
    print(f"   • Registos totais: {count:,}")
    print(f"   • CODEUT únicos: {codeut_unicos:,}")
    print(f"   • Ficheiros importados: {num_ficheiros}")
    print(f"   • Data mais antiga (FCARGA): {data_min}")
    print(f"   • Data mais recente (FCARGA): {data_max}")
    print()

print(f"📈 TOTAL DE REGISTOS NA BD: {total_geral:,}")
print()

con.close()

INSPEÇÃO DA BASE DE DADOS

Anos com tabelas na BD: 2024, 2025, 2026

📊 inform_26_2024:
   • Registos totais: 27,388
   • CODEUT únicos: 27,388
   • Ficheiros importados: 3
   • Data mais antiga (FCARGA): 20240101
   • Data mais recente (FCARGA): 20241231

📊 inform_26_2025:
   • Registos totais: 23,929
   • CODEUT únicos: 23,929
   • Ficheiros importados: 3
   • Data mais antiga (FCARGA): 20250101
   • Data mais recente (FCARGA): 20251231

📊 inform_26_2026:
   • Registos totais: 5,448
   • CODEUT únicos: 5,448
   • Ficheiros importados: 1
   • Data mais antiga (FCARGA): 20260101
   • Data mais recente (FCARGA): 20260731

📈 TOTAL DE REGISTOS NA BD: 56,765



In [13]:
# ==============================================================================
# FICHEIROS IMPORTADOS (POR TABELA)
# ==============================================================================

print("=" * 70)
print("FICHEIROS IMPORTADOS")
print("=" * 70)
print()

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

anos_na_bd = detectar_anos_na_bd(DB_PATH)

for ano in anos_na_bd:
    tabela = f"inform_26_{ano}"
    cur.execute(
        f'SELECT ficheiro_origem, COUNT(*) FROM "{tabela}" '
        f'GROUP BY ficheiro_origem ORDER BY COUNT(*) DESC'
    )
    resultado = cur.fetchall()

    if resultado:
        print(f"📁 inform_26_{ano}:")
        for ficheiro, count in resultado:
            print(f"   • {ficheiro}: {count:,} linhas")
    else:
        print(f"📁 inform_26_{ano}: (vazio)")
    print()

con.close()

FICHEIROS IMPORTADOS

📁 inform_26_2024:
   • SAL_DAT026 (7).xlsx: 16,978 linhas
   • SAL_DAT026 (8).xlsx: 10,405 linhas
   • SAL_DAT026 (6).xlsx: 5 linhas

📁 inform_26_2025:
   • SAL_DAT026 (4).xlsx: 14,209 linhas
   • SAL_DAT026 (5).xlsx: 9,690 linhas
   • SAL_DAT026 (3).xlsx: 30 linhas

📁 inform_26_2026:
   • SAL_DAT026 (2).xlsx: 5,448 linhas



## PASSO 6 — Ferramentas Interativas de Consulta

In [14]:
# ==============================================================================
# FERRAMENTA 1: Buscar por CODEUT
# ==============================================================================

def buscar_por_codeut(codeut_procura):
    """Procura um CODEUT em todas as tabelas."""
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    cur = con.cursor()

    anos_na_bd = detectar_anos_na_bd(DB_PATH)
    encontrados = False

    for ano in anos_na_bd:
        tabela = f"inform_26_{ano}"
        cur.execute(
            f'SELECT id, FCARGA, TRANSPORTISTA, LOCORIGEN, '
            f'LOCDESTINO, INGRESOUT, COSTEUT, MARGENUT FROM "{tabela}" WHERE CODEUT = ?',
            (codeut_procura,)
        )
        registos = cur.fetchall()

        if registos:
            encontrados = True
            print(f"✓ Encontrado em inform_26_{ano}:")
            for reg in registos:
                print(f"  ID: {reg['id']}, Data: {reg['FCARGA']}, "
                      f"Transportista: {reg['TRANSPORTISTA']}")
                print(f"  Rota: {reg['LOCORIGEN']} → {reg['LOCDESTINO']}")
                print(f"  Ingreso: {reg['INGRESOUT']}, Coste: {reg['COSTEUT']}, "
                      f"Margen: {reg['MARGENUT']}")
                print()

    con.close()

    if not encontrados:
        print(f"✗ CODEUT '{codeut_procura}' não encontrado em nenhuma tabela.")

# Exemplo: descomentar para testar
# buscar_por_codeut("123456")

print("✓ Função buscar_por_codeut() pronta")

✓ Função buscar_por_codeut() pronta


In [15]:
# ==============================================================================
# FERRAMENTA 2: Listar transportistas únicos
# ==============================================================================

def listar_transportistas(ano):
    """Lista transportistas únicos de um ano."""
    anos_na_bd = detectar_anos_na_bd(DB_PATH)

    if ano not in anos_na_bd:
        print(f"✗ Ano '{ano}' não existe. Anos disponíveis: {', '.join(anos_na_bd)}")
        return

    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    tabela = f"inform_26_{ano}"
    cur.execute(
        f'SELECT DISTINCT TRANSPORTISTA FROM "{tabela}" '
        f'WHERE TRANSPORTISTA IS NOT NULL AND TRANSPORTISTA != "" '
        f'ORDER BY TRANSPORTISTA'
    )
    transportistas = [row[0] for row in cur.fetchall()]
    con.close()

    print(f"✓ Transportistas únicos em inform_26_{ano}: {len(transportistas)}")
    print()
    for i, t in enumerate(transportistas[:20], 1):
        print(f"  {i}. {t}")
    if len(transportistas) > 20:
        print(f"  ... e mais {len(transportistas) - 20}")

# Exemplo: descomentar para testar
# listar_transportistas("2026")

print("✓ Função listar_transportistas() pronta")

✓ Função listar_transportistas() pronta


In [16]:
# ==============================================================================
# FERRAMENTA 3: Top 10 locais de origem
# ==============================================================================

def top_locais_origem(ano):
    """Mostra os top 10 locais de origem de um ano."""
    anos_na_bd = detectar_anos_na_bd(DB_PATH)

    if ano not in anos_na_bd:
        print(f"✗ Ano '{ano}' não existe. Anos disponíveis: {', '.join(anos_na_bd)}")
        return

    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    tabela = f"inform_26_{ano}"
    cur.execute(
        f'SELECT LOCORIGEN, COUNT(*) FROM "{tabela}" '
        f'WHERE LOCORIGEN IS NOT NULL AND LOCORIGEN != "" '
        f'GROUP BY LOCORIGEN ORDER BY COUNT(*) DESC LIMIT 10'
    )
    registos = cur.fetchall()
    con.close()

    print(f"✓ Top 10 locais de origem em inform_26_{ano}:")
    print()
    for i, (localidade, count) in enumerate(registos, 1):
        print(f"  {i}. {localidade}: {count:,} registos")

# Exemplo: descomentar para testar
# top_locais_origem("2026")

print("✓ Função top_locais_origem() pronta")

✓ Função top_locais_origem() pronta


## 📝 Notas Finais

In [17]:
print("=" * 70)
print("✓ NOTEBOOK COMPLETO EXECUTADO COM SUCESSO!")
print("=" * 70)
print()
print("📊 O que foi feito:")
print("  1. Detectou automaticamente os anos nos ficheiros (.xlsx)")
print("  2. Criou tabelas inform_26_2024 / inform_26_2025 / inform_26_2026")
print("  3. Inseriu dados sem duplicatas (chave: CODEUT)")
print("  4. Mostrou estatísticas completas")
print()
print("🛠️  Ferramentas disponíveis:")
print("  • buscar_por_codeut(codeut)   - Procurar por código de UT")
print("  • listar_transportistas(ano)  - Ver transportistas")
print("  • top_locais_origem(ano)      - Ver top 10 locais de origem")
print()
print("🚀 Próximos passos:")
print("  • Coloca ficheiros .xlsx novos na pasta e corre novamente")
print("  • Se tiveres dados de 2024/2025, serão criadas tabelas automaticamente")
print("  • Usa as funções para explorar os dados")
print()
print("=" * 70)

✓ NOTEBOOK COMPLETO EXECUTADO COM SUCESSO!

📊 O que foi feito:
  1. Detectou automaticamente os anos nos ficheiros (.xlsx)
  2. Criou tabelas inform_26_2024 / inform_26_2025 / inform_26_2026
  3. Inseriu dados sem duplicatas (chave: CODEUT)
  4. Mostrou estatísticas completas

🛠️  Ferramentas disponíveis:
  • buscar_por_codeut(codeut)   - Procurar por código de UT
  • listar_transportistas(ano)  - Ver transportistas
  • top_locais_origem(ano)      - Ver top 10 locais de origem

🚀 Próximos passos:
  • Coloca ficheiros .xlsx novos na pasta e corre novamente
  • Se tiveres dados de 2024/2025, serão criadas tabelas automaticamente
  • Usa as funções para explorar os dados

